# 01 — EDA: Model Perencana Makan

**Dataset:**
- `nutrition.csv` (1,346 makanan Indonesia) — **primary**
- `diet_recommendations_dataset.csv` (1,000 pasien) — **validation rules**

**Tujuan:**
- Distribusi makro (kalori/protein/karbo/lemak)
- Identifikasi 0-kalori (edge case → impute)
- Analisis nama makanan untuk heuristic kategorisasi
- **Cuisine clustering** (Padang/Jawa/Sunda heuristic) — riset IPB Jurnal 2024
- Distribusi harga per kategori (validasi heuristic pricing)
- Eksplorasi dataset validasi medis (Diabetes/Hypertension)

**Riset acuan:**
- Frontiers Nutrition 2025: GA-based Indonesian restaurant food selection
- IPB Jurnal Keteknikan 2024: Indonesian food recommendation untuk NCD

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json

sns.set_theme(style='darkgrid')
os.makedirs('output/eda', exist_ok=True)

NUT_PATH  = '../../dataset/Model_Perencana Makan_dan_Nutrisi/Indonesian Food & Drink Nutrition Dataset/nutrition.csv'
DIET_PATH = '../../dataset/Model_Perencana Makan_dan_Nutrisi/Diet Recommendations Dataset/diet_recommendations_dataset.csv'

df = pd.read_csv(NUT_PATH)
print(f'Nutrition dataset: {df.shape}')
df.head()

In [ ]:
# ── Basic Stats ──
print('Kolom:', list(df.columns))
print('\nMissing values:')
print(df.isnull().sum())
print('\nStatistik:')
df[['calories','proteins','fat','carbohydrate']].describe().round(2)

In [ ]:
# ── Distribusi Makro ──
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
macro_cols = {'calories':'Kalori (kcal)', 'proteins':'Protein (g)',
              'fat':'Lemak (g)', 'carbohydrate':'Karbohidrat (g)'}

for ax, (col, label) in zip(axes.flatten(), macro_cols.items()):
    data = df[col].dropna()
    ax.hist(data, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
    ax.axvline(data.median(), color='orange', linestyle='--', label=f'Med: {data.median():.1f}')
    ax.axvline(data.mean(), color='red', linestyle=':', label=f'Mean: {data.mean():.1f}')
    ax.set_title(label, fontweight='bold')
    ax.legend(fontsize=9)

plt.suptitle('Distribusi Makronutrien — Indonesian Food', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('output/eda/macro_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 0-Kalori Detection ──
zero_cal = df[df['calories'] == 0]
print(f'Entri 0 kalori: {len(zero_cal)} ({len(zero_cal)/len(df)*100:.1f}%)')
print('Sample (akan diimpute ke 1 di preprocessing):')
print(zero_cal[['name','calories','proteins','fat','carbohydrate']].head(10))

In [ ]:
# ── Kategorisasi Heuristic (Enhanced) ──
# Keyword dictionary 3x lebih kaya dari paper IPB Jurnal Keteknikan 2024
KEYWORDS = {
    'PROTEIN':    ['ayam','sapi','ikan','telur','tahu','tempe','daging','udang','cumi',
                   'bebek','kambing','lele','bandeng','gurame','salmon','tuna','bakso',
                   'keju','sosis ayam','kornet','sarden','ati','hati','jerohan'],
    'VEGETABLE':  ['sayur','bayam','kangkung','brokoli','wortel','sawi','kol','daun',
                   'buncis','kubis','terong','labu','timun','selada','bawang','tomat',
                   'gado-gado','urap','pecel','asinan','karedok','lalap'],
    'FRUIT':      ['buah','apel','mangga','pisang','jeruk','semangka','anggur','pepaya',
                   'salak','melon','nanas','alpukat','lemon','jambu','rambutan','duku',
                   'durian','mangosteen','manggis','kelengkeng','sirsak','markisa'],
    'BEVERAGE':   ['minum','teh','kopi','jus','susu','air ','minuman','sirup','soda',
                   'es teh','es jeruk','cappuccino','latte','smoothie','milkshake',
                   'wedang','jamu','bandrek','sekoteng','cendol'],
    'DESSERT':    ['kue','puding','es krim','dodol','klepon','bakpia','cake','martabak',
                   'brownie','coklat','permen','gulali','halua','tart','onde','getuk',
                   'wajik','lemper','lapis','bika ambon'],
    'SNACK':      ['keripik','biskuit','kerupuk','nugget','sosis','chiki','crackers',
                   'gorengan','bakwan','pisang goreng','singkong goreng','tahu isi'],
    'STAPLE':     ['nasi','mie','kentang','jagung','roti','pasta','ubi','singkong',
                   'bihun','ketupat','lontong','bubur','oatmeal','sereal','spaghetti'],
}

def categorize(name):
    n = str(name).lower()
    for cat in ['PROTEIN','VEGETABLE','FRUIT','BEVERAGE','DESSERT','SNACK','STAPLE']:
        if any(kw in n for kw in KEYWORDS[cat]):
            return cat
    return 'STAPLE'  # Default

df['category_guess'] = df['name'].apply(categorize)
cat_counts = df['category_guess'].value_counts()

plt.figure(figsize=(10, 5))
cat_counts.plot(kind='bar', color=sns.color_palette('viridis', len(cat_counts)))
plt.title('Distribusi Kategori (Heuristic Enhanced)', fontweight='bold')
plt.xlabel('Kategori'); plt.ylabel('Jumlah')
plt.xticks(rotation=30)
for i, v in enumerate(cat_counts.values):
    plt.text(i, v+5, str(v), ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('output/eda/category_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(cat_counts)

In [ ]:
# ── Cuisine Clustering (Padang/Jawa/Sunda/Asian/Western) ──
CUISINE_KEYWORDS = {
    'PADANG':   ['rendang','dendeng','sate padang','gulai','kalio','lemang','sambal lado',
                  'gajeboh','asam pedas','pangek'],
    'JAVA':     ['gudeg','soto','rawon','nasi pecel','tumpeng','wedang','klepon','onde',
                  'getuk','tahu petis','rujak cingur','lontong balap'],
    'SUNDA':    ['karedok','lalap','nasi tutug','nasi liwet','batagor','siomay','pepes',
                  'sayur asem','ulukutek'],
    'ASIAN':    ['ramen','sushi','dimsum','siomay','bakpao','mie ayam','kwetiau','nasi goreng'],
    'WESTERN':  ['burger','pizza','steak','spaghetti','sandwich','pasta','french fries']
}

def guess_cuisine(name):
    n = str(name).lower()
    for cuisine, kws in CUISINE_KEYWORDS.items():
        if any(kw in n for kw in kws):
            return cuisine
    return 'GENERAL_INDO'  # Default Indonesian

df['cuisine_guess'] = df['name'].apply(guess_cuisine)
cuisine_counts = df['cuisine_guess'].value_counts()

print('Distribusi cuisine (heuristic):')
print(cuisine_counts)

In [ ]:
# ── Heuristic Price Validation ──
BASE_PRICE = {
    'STAPLE': 8000, 'PROTEIN': 18000, 'VEGETABLE': 6000, 'FRUIT': 7000,
    'BEVERAGE': 5000, 'DESSERT': 10000, 'SNACK': 5000
}

def estimate_price(row):
    base = BASE_PRICE.get(row['category_guess'], 7000)
    cal = max(row['calories'], 10)
    return round((base * (1 + cal / 500)) / 500) * 500

df['est_price_idr'] = df.apply(estimate_price, axis=1)

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df, x='category_guess', y='est_price_idr', palette='viridis', ax=ax)
ax.set_title('Distribusi Estimasi Harga per Kategori (IDR)', fontweight='bold')
ax.set_xlabel('Kategori'); ax.set_ylabel('Estimated Price (IDR)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('output/eda/price_per_category.png', dpi=150)
plt.show()

print('Harga per kategori:')
print(df.groupby('category_guess')['est_price_idr'].describe().round(0))

In [ ]:
# ── Diet Recommendations Dataset Exploration ──
try:
    df_diet = pd.read_csv(DIET_PATH)
    print(f'Diet dataset: {df_diet.shape}')
    print('Kolom:', list(df_diet.columns))
    
    print('\nDiet_Recommendation distribusi:')
    print(df_diet['Diet_Recommendation'].value_counts())
    print('\nDisease_Type distribusi:')
    print(df_diet['Disease_Type'].value_counts())
    
    # Cross-tab Disease vs Diet
    print('\nCross-tab Disease × Diet:')
    print(pd.crosstab(df_diet['Disease_Type'], df_diet['Diet_Recommendation']))
except Exception as e:
    print(f'Diet dataset error: {e}')

In [ ]:
# ── Save Summary ──
summary = {
    'n_foods': len(df),
    'zero_calorie_count': int(len(zero_cal)),
    'calorie_range': f"{df['calories'].min()} – {df['calories'].max()}",
    'avg_calories': round(float(df['calories'].mean()), 1),
    'category_dist': cat_counts.to_dict(),
    'cuisine_dist': cuisine_counts.to_dict(),
    'price_summary_per_category': df.groupby('category_guess')['est_price_idr'].mean().round(0).to_dict(),
    'augmentation_columns_required': ['category', 'estimated_price_idr', 'is_halal',
                                       'is_vegetarian', 'is_vegan', 'is_gluten_free', 'cuisine'],
}
with open('output/eda/eda_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('✅ EDA selesai.')
print(json.dumps(summary, indent=2))